# NN Architecture 2C: Bidirectional LSTM for Rayleigh Fading

**Reference**: "Classification of Stochastic Systems with Deep Learning and Hypothesis Testing" (project paper)

**Approach**: LSTM networks capture temporal dependencies in stochastic processes, ideal for Rayleigh fading channels

**Rationale**: 
- Rayleigh fading channel is a stochastic process (time-varying gain)
- LSTM can learn dynamic relationships: $h(t) = f(h(t-1), \text{channel state})$
- Bidirectional LSTM uses both forward and backward information
- Gating mechanism (forget, input, output gates) learns what to remember

**Architecture**:
```
Input: [τ_correlator, h_est, SNR, energy] → reshape as sequence (4D time steps)
    ↓
Bidirectional LSTM(64, return_sequences=True) → Dropout(0.3)
    ↓
Bidirectional LSTM(32) → Dropout(0.2)
    ↓
Dense(32, ReLU)
    ↓
Dense(1, Sigmoid) → Binary output
```

**Framework**: PyTorch with LSTM cells

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# SETUP PATHS FOR NEW DIRECTORY STRUCTURE
# ============================================================================
from pathlib import Path

notebook_dir = Path.cwd()  # Current: notebooks/
project_root = notebook_dir.parent  # Go up to: Redes Neurais/
results_dir = project_root / "results"
data_dir = results_dir / "data"
models_dir = results_dir / "models"
visualizations_dir = results_dir / "visualizations"

models_dir.mkdir(parents=True, exist_ok=True)
visualizations_dir.mkdir(parents=True, exist_ok=True)

with h5py.File(str(data_dir / 'dataset_nn_100k.h5'), 'r') as f:
    X_train = f['X_train'][:]
    y_train = f['y_train'][:]
    X_val = f['X_val'][:]
    y_val = f['y_val'][:]
    X_test = f['X_test'][:]
    y_test = f['y_test'][:]

print(f"✓ Dataset loaded successfully")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train)
y_train = torch.LongTensor(y_train)
X_val = torch.FloatTensor(X_val)
y_val = torch.LongTensor(y_val)
X_test = torch.FloatTensor(X_test)
y_test = torch.LongTensor(y_test)


In [ ]:
        torch.save(model.state_dict(), str(models_dir / 'model_lstm_best.pth'))
        best_val_loss = val_loss

print("\n✓ Training complete!")

# ==============================================================================
# EVALUATION & VISUALIZATION
# ==============================================================================

model.eval()
with torch.no_grad():
    y_train_pred_proba = model(X_train).numpy()
    y_val_pred_proba = model(X_val).numpy()
    y_test_pred_proba = model(X_test).numpy()

y_train_pred = (y_train_pred_proba > 0.5).astype(int).flatten()
y_val_pred = (y_val_pred_proba > 0.5).astype(int).flatten()
y_test_pred = (y_test_pred_proba > 0.5).astype(int).flatten()

print("\nTest Set Performance:")
print(f"  Accuracy: {np.mean(y_test_pred == y_test.numpy()):.4f}")
from sklearn.metrics import confusion_matrix, roc_auc_score
cm = confusion_matrix(y_test.numpy(), y_test_pred)
print(f"  Confusion Matrix:\n{cm}")
print(f"  AUC: {roc_auc_score(y_test.numpy(), y_test_pred_proba):.4f}")

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Training history
axes[0, 0].plot(train_losses, label='Train', linewidth=2)
axes[0, 0].plot(val_losses, label='Validation', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('LSTM: Training History (Rayleigh)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Confusion matrix
im = axes[0, 1].imshow(cm, cmap='Blues', interpolation='nearest')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('True')
axes[0, 1].set_title('Confusion Matrix (Test Set)')
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_yticks([0, 1])
for i in range(2):
    for j in range(2):
        axes[0, 1].text(j, i, str(cm[i, j]), ha='center', va='center', color='white', fontsize=14)

# Output distribution
axes[1, 0].hist(y_test_pred_proba[y_test==0], bins=30, alpha=0.6, label='H0 (Fraudulent)', color='blue')
axes[1, 0].hist(y_test_pred_proba[y_test==1], bins=30, alpha=0.6, label='H1 (Authentic)', color='orange')
axes[1, 0].axvline(0.5, color='red', linestyle='--', linewidth=2, label='Decision threshold')
axes[1, 0].set_xlabel('LSTM Output Probability')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('LSTM Output Distribution')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# ROC Curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y_test.numpy(), y_test_pred_proba)
auc = roc_auc_score(y_test.numpy(), y_test_pred_proba)
axes[1, 1].plot(fpr, tpr, linewidth=2.5, label=f'LSTM (AUC={auc:.4f})')
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random Classifier')
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].set_title('ROC Curve')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
output_file = visualizations_dir / 'results_lstm.png'
plt.savefig(str(output_file), dpi=100, bbox_inches='tight')
plt.show()
